# Hard Benchmark Notebook — Where Deep-MELU Actually Differs

This notebook addresses the core problem: on easy datasets all methods score ~1.0, making comparison meaningless.

**Four sections:**
- **A** — Harder dataset designs where methods genuinely differ
- **B** — Contamination robustness sweep (Deep-MELU's MCD advantage)
- **C** — Controlled experiments: vary one factor, freeze everything else
- **D** — AUCPR curves — more discriminative than AUROC at low contamination

These are the experiments that belong in the Pattern Recognition paper.

## Cell 1 — Imports and model definitions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.special import betainc
from scipy.stats import chi2, t as t_dist
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

# ── Student-t CDF ─────────────────────────────────────────────────────────────
def _tcdf(x, nu=4.0):
    nu = max(float(nu), 2.0)
    z  = nu / (nu + np.clip(x**2, 1e-30, None))
    ib = betainc(nu/2, 0.5, np.clip(z, 1e-12, 1-1e-12))
    return np.where(x >= 0, 1.0-ib/2.0, ib/2.0)

def safe_chol_inv(cov, lat, n):
    d = max(np.diag(cov).max(), 1e-8)
    r = n / max(lat, 1)
    base_eps = d * max(1e-3, min(0.1, 5.0/max(r,1)))
    for eps in [base_eps, base_eps*10, base_eps*100, d*0.5]:
        try:
            L = np.linalg.cholesky(cov + eps*np.eye(lat))
            Li = np.linalg.inv(L)
            if not np.isnan(Li).any() and np.linalg.cond(Li) < 1e7:
                return Li
        except np.linalg.LinAlgError:
            continue
    return np.eye(lat)

class DeepMELU:
    def __init__(self, dim, hidden=64, latent=32,
                 alpha=1.0, beta=0.4, nu=4.0, momentum=0.95):
        self.dim=dim; self.latent=latent
        self.alpha=alpha; self.beta=beta; self.nu=nu; self.momentum=momentum
        np.random.seed(0)
        self.W1 = np.random.randn(dim,hidden)   * np.sqrt(2/dim)
        self.W2 = np.random.randn(hidden,latent) * np.sqrt(2/hidden)
        self.Wd = np.random.randn(latent,dim)    * np.sqrt(2/latent)
        self.mu=np.zeros(latent); self.Li=np.eye(latent); self.tau=1.0
        self.mu_d=0.; self.sd_d=1.; self.mu_e=0.; self.sd_e=1.; self.thr=1.

    def _sw(self,x): return x/(1+np.exp(-np.clip(x,-50,50)))
    def _enc(self,X):
        return np.nan_to_num(self._sw(self._sw(X@self.W1)@self.W2))

    def _melu(self,H):
        T1 = H*_tcdf(H,self.nu)
        c  = H-self.mu; w = np.nan_to_num(c@self.Li.T)
        m  = np.sqrt(np.maximum((w**2).sum(1),0))
        gate = (m>=self.tau).astype(float)[:,None]
        amp  = self.alpha*np.sign(H)*(np.exp(np.clip(self.beta*(m[:,None]-self.tau),-20,20))-1)
        return T1+gate*amp

    def _mcd(self,Z,h_frac=0.75):
        n=len(Z); h=max(int(n*h_frac),self.latent+1)
        best_det,best_mu,best_cov=np.inf,None,None
        for _ in range(8):
            idx=np.random.choice(n,h,replace=False); sub=Z[idx]
            for _ in range(6):
                mu=sub.mean(0); d=sub-mu
                cov=d.T@d/max(len(sub)-1,1)+1e-4*np.eye(self.latent)
                Si=np.linalg.inv(cov)
                ds=np.sqrt(np.maximum(np.einsum('bi,ij,bj->b',Z-mu,Si,Z-mu),0))
                idx=np.argsort(ds)[:h]; sub=Z[idx]
            mu=sub.mean(0); d=sub-mu; cov=d.T@d/max(len(sub)-1,1)
            det=np.linalg.det(cov+1e-4*np.eye(self.latent))
            if det<best_det: best_det=det; best_mu=mu; best_cov=cov
        return best_mu, best_cov

    def fit(self,X_train,n_epochs=60,lr=0.005,batch=64):
        n=len(X_train)
        for ep in range(n_epochs):
            Z=self._enc(X_train)
            mu,cov=self._mcd(Z)
            self.mu=mu; self.Li=safe_chol_inv(cov,self.latent,len(Z))
            c=Z-mu; w=np.nan_to_num(c@self.Li.T)
            dm=np.sqrt(np.maximum((w**2).sum(1),0))
            self.tau=self.momentum*self.tau+(1-self.momentum)*dm.mean()
            idx=np.random.permutation(n)
            for i in range(0,n,batch):
                xb=X_train[idx[i:i+batch]]
                zb=self._enc(xb); zm=np.nan_to_num(self._melu(zb))
                xh=zm@self.Wd; err=xh-xb
                self.Wd-=lr*np.clip(zm.T@err/max(len(xb),1),-1,1)
        Z=self._enc(X_train); Zm=np.nan_to_num(self._melu(Z))
        Xh=Zm@self.Wd; c=Z-self.mu
        dm=np.sqrt(np.maximum((np.nan_to_num(c@self.Li.T)**2).sum(1),0))
        er=np.abs(X_train-Xh).mean(1)
        self.mu_d=dm.mean(); self.sd_d=max(dm.std(),1e-6)
        self.mu_e=er.mean(); self.sd_e=max(er.std(),1e-6)
        sd=np.maximum(0,(dm-self.mu_d)/self.sd_d)
        se=np.maximum(0,(er-self.mu_e)/self.sd_e)
        self.thr=np.percentile(0.5*sd+0.5*se,95)
        return self

    def score(self,X):
        Z=self._enc(X); Zm=np.nan_to_num(self._melu(Z)); Xh=Zm@self.Wd
        c=Z-self.mu
        dm=np.sqrt(np.maximum((np.nan_to_num(c@self.Li.T)**2).sum(1),0))
        er=np.abs(X-Xh).mean(1)
        return 0.5*np.maximum(0,(dm-self.mu_d)/self.sd_d)+0.5*np.maximum(0,(er-self.mu_e)/self.sd_e)

def run_all(X,y,n_epochs=50):
    sc=StandardScaler(); X_sc=sc.fit_transform(X); X_in=X_sc[y==0]
    res={}
    for name,fn in [
        ("Deep-MELU",   lambda: DeepMELU(X.shape[1]).fit(X_in,n_epochs=n_epochs).score(X_sc)),
        ("IsoForest",   lambda: -IsolationForest(200,contamination="auto",random_state=42).fit(X_in).score_samples(X_sc)),
        ("LOF",         lambda: -LocalOutlierFactor(20,novelty=True,contamination="auto").fit(X_in).score_samples(X_sc)),
        ("OC-SVM",      lambda: -OneClassSVM(kernel="rbf",nu=0.1,gamma="scale").fit(X_in).score_samples(X_sc)),
    ]:
        try:
            sc_raw=fn()
            if np.isnan(sc_raw).any(): raise ValueError("NaN")
            res[name]=dict(
                auroc=float(roc_auc_score(y,sc_raw)),
                aucpr=float(average_precision_score(y,sc_raw)),
                scores=sc_raw)
        except Exception as e:
            res[name]=dict(auroc=0.5,aucpr=0.0,scores=np.zeros(len(X)))
    return res

COLORS={"Deep-MELU":"#1D9E75","IsoForest":"#534AB7","LOF":"#BA7517","OC-SVM":"#888780"}
print("All dependencies loaded. DeepMELU defined.")


---
## Section A — Hard Dataset Designs

These 6 datasets are specifically designed to **not** be trivially separable. Each exposes a different structural property that methods handle differently.

## Cell 2 — Define 6 hard datasets

In [ ]:
def make_local_anomalies(n=600, dim=8, cont=0.05, seed=42):
    """
    LOCAL anomalies — overlap with inlier distribution.
    Hardest type: outliers are inside the inlier cloud boundary
    but deviate from local neighbourhood density.
    """
    np.random.seed(seed)
    n_out=max(1,int(n*cont)); n_in=n-n_out
    cov=np.eye(dim)*0.5+0.3*np.random.randn(dim,dim)@np.random.randn(dim,dim).T/dim
    cov=(cov+cov.T)/2+np.eye(dim)*0.2
    X_in=np.random.multivariate_normal(np.zeros(dim),cov,n_in)
    # Local outliers: scale covariance by 5x — they look like inliers globally
    X_out=np.random.multivariate_normal(np.zeros(dim),cov*5,n_out)
    X=np.vstack([X_in,X_out]); y=np.array([0]*n_in+[1]*n_out)
    return X,y,"local_anomalies","Local outliers (Σ×5, same centre)"

def make_dependency_anomalies(n=600, dim=8, cont=0.05, seed=42):
    """
    DEPENDENCY anomalies — normal marginals, broken correlations.
    Feature values are individually normal but joint distribution is wrong.
    Euclidean/IsoForest completely blind. Mahalanobis catches it.
    """
    np.random.seed(seed)
    n_out=max(1,int(n*cont)); n_in=n-n_out
    rho=0.85
    cov=np.array([[rho**abs(i-j) for j in range(dim)] for i in range(dim)])
    X_in=np.random.multivariate_normal(np.zeros(dim),cov,n_in)
    # Outliers: each dimension is individually standard normal but UNCORRELATED
    # Their feature values look fine; only the joint structure is wrong
    X_out=np.random.randn(n_out,dim)  # iid standard normal, no correlation
    X=np.vstack([X_in,X_out]); y=np.array([0]*n_in+[1]*n_out)
    return X,y,"dependency_anomalies","Broken correlations (normal margins, wrong joint)"

def make_masked_anomalies(n=600, dim=8, cont=0.15, mask_factor=3, seed=42):
    """
    MASKED anomalies — multiple clustered outliers that corrupt covariance.
    Standard covariance shifts toward the outlier cluster (masking).
    MCD-based methods are specifically designed to handle this.
    """
    np.random.seed(seed)
    n_out=max(1,int(n*cont)); n_in=n-n_out
    X_in=np.random.randn(n_in,dim)
    # Outliers form a TIGHT CLUSTER far away
    center=np.random.randn(dim)*4
    X_out=np.random.randn(n_out,dim)*0.3+center
    X=np.vstack([X_in,X_out]); y=np.array([0]*n_in+[1]*n_out)
    return X,y,"masked_anomalies","Clustered outliers — masking test (15% contamination)"

def make_high_dim_sparse(n=500, dim=50, cont=0.08, seed=42):
    """
    HIGH-DIM SPARSE — outliers deviate in only 3 of 50 dimensions.
    Euclidean distance diluted by 47 normal dimensions.
    Mahalanobis in the right projection finds it.
    """
    np.random.seed(seed)
    n_out=max(1,int(n*cont)); n_in=n-n_out
    X_in=np.random.randn(n_in,dim)
    X_out=np.random.randn(n_out,dim)
    hot=np.random.choice(dim,3,replace=False)
    X_out[:,hot]+=np.random.choice([-1,1],(n_out,3))*5
    X=np.vstack([X_in,X_out]); y=np.array([0]*n_in+[1]*n_out)
    return X,y,"high_dim_sparse",f"High-dim (d={dim}), outliers in 3 of {dim} dims"

def make_low_contamination(n=1000, dim=10, cont=0.01, seed=42):
    """
    VERY LOW CONTAMINATION — 1% outliers.
    AUROC is meaningless here; AUCPR is the right metric.
    Tests precision at very high rank thresholds.
    """
    np.random.seed(seed)
    n_out=max(1,int(n*cont)); n_in=n-n_out
    X_in=np.random.randn(n_in,dim)
    X_out=np.random.randn(n_out,dim)*1.5+np.random.choice([-1,1],dim)*3
    X=np.vstack([X_in,X_out]); y=np.array([0]*n_in+[1]*n_out)
    return X,y,"low_contamination",f"Very low contamination (1% = {n_out} outliers)"

def make_concept_drift(n=800, dim=8, cont=0.08, seed=42):
    """
    CONCEPT DRIFT — inlier distribution shifts midway through data.
    EMA tau adapts; fixed-threshold methods fail on the second half.
    Tests Deep-MELU's adaptive threshold advantage.
    """
    np.random.seed(seed)
    n_out=max(1,int(n*cont)); n_in=n-n_out
    half=n_in//2
    # First half: centred at 0
    X1=np.random.randn(half,dim)
    # Second half: centre shifts to 2
    X2=np.random.randn(n_in-half,dim)+2.0
    X_in=np.vstack([X1,X2])
    # Outliers: extreme in the direction perpendicular to drift
    X_out=np.zeros((n_out,dim))
    for i in range(n_out):
        t=np.random.rand()
        base=np.zeros(dim); base[0]=t*2  # drift direction
        X_out[i]=base+np.random.randn(dim)*0.5
        X_out[i,1]+=np.random.choice([-1,1])*5  # anomaly in dim 1
    X=np.vstack([X_in,X_out]); y=np.array([0]*n_in+[1]*n_out)
    return X,y,"concept_drift","Inlier distribution drifts mid-sequence"

HARD_DATASETS = [
    make_local_anomalies(),
    make_dependency_anomalies(),
    make_masked_anomalies(),
    make_high_dim_sparse(),
    make_low_contamination(),
    make_concept_drift(),
]

print("Hard datasets defined:")
for X,y,name,desc in HARD_DATASETS:
    n_out=y.sum()
    print(f"  {name:<25}  n={len(X):4d}  dim={X.shape[1]:2d}  "
          f"outliers={n_out:3d} ({n_out/len(X)*100:.1f}%)  — {desc}")


## Cell 3 — Run all methods on all hard datasets

> Expected runtime: ~10–20 minutes

In [ ]:
hard_results = {}

for X, y, name, desc in HARD_DATASETS:
    print(f"\n{'='*55}")
    print(f"  {name}  ({desc})")
    print(f"{'='*55}")
    hard_results[name] = {"desc": desc}
    res = run_all(X, y, n_epochs=60)
    hard_results[name].update(res)
    for method, m in res.items():
        print(f"  {method:<14}  AUROC={m['auroc']:.3f}  AUCPR={m['aucpr']:.3f}")

print("\n✓ Hard dataset benchmarks complete.")


## Cell 4 — Hard dataset comparison chart

In [ ]:
HARD_NAMES = [X[2] for X in HARD_DATASETS]
METHODS_H = ["Deep-MELU","IsoForest","LOF","OC-SVM"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Hard Dataset Benchmarks — AUROC (top) and AUCPR (bottom)", fontsize=13)
axes = axes.flatten()

for i, (X, y, name, desc) in enumerate(HARD_DATASETS):
    ax = axes[i]
    methods = list(METHODS_H)
    aurocs  = [hard_results[name][m]["auroc"] for m in methods]
    aucprs  = [hard_results[name][m]["aucpr"] for m in methods]
    x_pos   = np.arange(len(methods))
    bars    = ax.bar(x_pos-0.2, aurocs, width=0.35,
                     color=[COLORS[m] for m in methods], alpha=0.9, label="AUROC")
    bars2   = ax.bar(x_pos+0.2, aucprs, width=0.35,
                     color=[COLORS[m] for m in methods], alpha=0.45, label="AUCPR",
                     hatch="//")
    ax.set_xticks(x_pos); ax.set_xticklabels(methods, fontsize=8, rotation=15)
    ax.set_ylim(0, 1.15); ax.set_title(f"{name.replace('_',' ')}", fontsize=10)
    ax.axhline(0.5, color="gray", lw=0.8, ls="--", alpha=0.4)
    ax.grid(axis="y", alpha=0.3)
    # Annotate best method
    best_auroc = max(zip(methods, aurocs), key=lambda x: x[1])
    ax.text(0.5, 1.08, f"Best: {best_auroc[0]} ({best_auroc[1]:.3f})",
            ha="center", transform=ax.transAxes, fontsize=8, color="#085041")
    for b, v in zip(bars, aurocs):
        ax.text(b.get_x()+b.get_width()/2, v+0.02,
                f"{v:.3f}", ha="center", va="bottom", fontsize=7)

# Legend
from matplotlib.patches import Patch
handles = [Patch(facecolor=COLORS[m], label=m) for m in METHODS_H]
handles += [Patch(facecolor="gray", alpha=0.5, hatch="//", label="AUCPR (hatched)")]
fig.legend(handles=handles, loc="lower center", ncol=5, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()
plt.savefig("outputs/hard_dataset_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → outputs/hard_dataset_results.png")


---
## Section B — Contamination Robustness Sweep

This is Deep-MELU's clearest structural advantage: the MCD estimator is robust to contamination up to 25%. Standard methods degrade as contamination increases.

## Cell 5 — Contamination sweep

In [ ]:
# Sweep contamination from 5% to 40% on a correlated dataset
# Deep-MELU (MCD) should maintain AUROC; standard methods should degrade

contam_levels = [0.05, 0.08, 0.10, 0.12, 0.15, 0.18, 0.20, 0.25, 0.30, 0.35, 0.40]
n_trials      = 8   # average over multiple seeds for stability
dim, n        = 10, 400

sweep_results = {m: {"auroc": [], "aucpr": []} for m in METHODS_H}

print("Running contamination sweep...")
print(f"{'Contamination':>14}", end="")
for m in METHODS_H: print(f"  {m:<12}", end="")
print()
print("-" * 70)

for cont in contam_levels:
    trial_auroc = {m: [] for m in METHODS_H}
    for seed in range(n_trials):
        np.random.seed(seed*100)
        n_out = max(1, int(n*cont)); n_in = n-n_out
        # Correlated inliers
        rho = 0.7
        cov = np.array([[rho**abs(i-j) for j in range(dim)] for i in range(dim)])
        X_in  = np.random.multivariate_normal(np.zeros(dim), cov, n_in)
        # Outliers deviate against correlation
        L = np.linalg.cholesky(cov)
        X_out = np.array([L@(np.random.randn(dim)*3*np.where(
            np.random.rand(dim)>0.5,1,-1)) for _ in range(n_out)])
        X = np.vstack([X_in, X_out])
        y = np.array([0]*n_in + [1]*n_out)
        res = run_all(X, y, n_epochs=40)
        for m in METHODS_H:
            trial_auroc[m].append(res[m]["auroc"])

    print(f"  {cont*100:5.0f}%        ", end="")
    for m in METHODS_H:
        avg = np.mean(trial_auroc[m])
        sweep_results[m]["auroc"].append(avg)
        print(f"  {avg:.3f}       ", end="")
    print()

print("\n✓ Contamination sweep complete.")


## Cell 6 — Plot contamination robustness

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Contamination Robustness — Deep-MELU (MCD) vs Baselines", fontsize=13)

x_cont = [c*100 for c in contam_levels]

# Panel 1: AUROC vs contamination
ax = axes[0]
for m in METHODS_H:
    lw = 2.8 if m == "Deep-MELU" else 1.5
    ls = "-" if m == "Deep-MELU" else "--"
    ax.plot(x_cont, sweep_results[m]["auroc"],
            color=COLORS[m], lw=lw, ls=ls, marker="o", ms=5, label=m)

ax.axvline(25, color="gray", lw=1, ls=":", alpha=0.7,
           label="MCD breakdown point (25%)")
ax.fill_betweenx([0.3, 1.05], 0, 25, alpha=0.04, color="#1D9E75",
                 label="MCD safe zone")
ax.set_xlabel("Contamination %"); ax.set_ylabel("AUROC")
ax.set_title("AUROC vs contamination level")
ax.legend(fontsize=9); ax.set_ylim(0.3, 1.05)
ax.grid(alpha=0.3)

# Panel 2: degradation relative to 5% baseline
ax = axes[1]
for m in METHODS_H:
    baseline = sweep_results[m]["auroc"][0]
    degradation = [baseline - v for v in sweep_results[m]["auroc"]]
    lw = 2.8 if m == "Deep-MELU" else 1.5
    ls = "-" if m == "Deep-MELU" else "--"
    ax.plot(x_cont, degradation, color=COLORS[m], lw=lw, ls=ls,
            marker="o", ms=5, label=m)

ax.axhline(0, color="black", lw=0.8)
ax.axvline(25, color="gray", lw=1, ls=":", alpha=0.7)
ax.fill_betweenx([-0.05, 0.5], 0, 25, alpha=0.04, color="#1D9E75")
ax.set_xlabel("Contamination %")
ax.set_ylabel("AUROC drop from 5% baseline")
ax.set_title("Performance degradation as contamination rises")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/contamination_robustness.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/contamination_robustness.png")


---
## Section C — Controlled Experiments

Vary one factor at a time. These are the cleanest experiments for the paper because they isolate exactly what drives performance differences.

## Cell 7 — Experiment 1: Correlation strength

In [ ]:
# As correlation ρ increases, Euclidean methods degrade; Mahalanobis-based stay strong
rho_values = [0.0, 0.2, 0.4, 0.6, 0.7, 0.8, 0.85, 0.90, 0.95]
n_trials_c = 10; n, dim, cont = 400, 8, 0.10
rho_results = {m: [] for m in METHODS_H}

print("Varying correlation strength ρ...")
for rho in rho_values:
    trial = {m: [] for m in METHODS_H}
    for seed in range(n_trials_c):
        np.random.seed(seed*77)
        n_out=max(1,int(n*cont)); n_in=n-n_out
        cov=np.array([[rho**abs(i-j) for j in range(dim)] for i in range(dim)])
        cov+=np.eye(dim)*0.01
        X_in=np.random.multivariate_normal(np.zeros(dim),cov,n_in)
        L=np.linalg.cholesky(cov)
        # Anti-correlated outliers: deviate against the dominant correlation direction
        X_out=np.array([L@(np.random.randn(dim)*2.5*np.where(
            np.random.rand(dim)>0.5,1,-1)) for _ in range(n_out)])
        X=np.vstack([X_in,X_out]); y=np.array([0]*n_in+[1]*n_out)
        res=run_all(X,y,n_epochs=40)
        for m in METHODS_H: trial[m].append(res[m]["auroc"])
    for m in METHODS_H: rho_results[m].append(np.mean(trial[m]))
    print(f"  ρ={rho:.2f}  " + "  ".join(f"{m[:5]}={np.mean(trial[m]):.3f}" for m in METHODS_H))

fig, ax = plt.subplots(figsize=(8, 5))
for m in METHODS_H:
    lw=2.8 if m=="Deep-MELU" else 1.5
    ls="-" if m=="Deep-MELU" else "--"
    ax.plot(rho_values, rho_results[m], color=COLORS[m], lw=lw, ls=ls,
            marker="o", ms=5, label=m)
ax.set_xlabel("Correlation strength ρ")
ax.set_ylabel("AUROC"); ax.set_ylim(0.3, 1.05)
ax.set_title("AUROC vs inter-feature correlation strength
"
             "(anti-correlated outliers — Mahalanobis advantage increases with ρ)")
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/correlation_experiment.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/correlation_experiment.png")


## Cell 8 — Experiment 2: Variance anisotropy ratio

In [ ]:
# As variance ratio σ_max/σ_min increases, Euclidean methods fail; Mahal. holds
ratios = [1, 2, 5, 10, 20, 50, 100]
n_trials_a = 10; n, dim, cont = 400, 8, 0.10
ani_results = {m: [] for m in METHODS_H}

print("Varying anisotropy ratio σ_max/σ_min...")
for ratio in ratios:
    trial = {m: [] for m in METHODS_H}
    for seed in range(n_trials_a):
        np.random.seed(seed*33)
        n_out=max(1,int(n*cont)); n_in=n-n_out
        scales=np.linspace(1, ratio, dim)
        X_in=np.random.randn(n_in,dim)*scales
        X_out=np.random.randn(n_out,dim)*scales
        low_d=np.argmin(scales)
        X_out[:,low_d]+=np.random.choice([-1,1],n_out)*scales[low_d]*5
        X=np.vstack([X_in,X_out]); y=np.array([0]*n_in+[1]*n_out)
        res=run_all(X,y,n_epochs=40)
        for m in METHODS_H: trial[m].append(res[m]["auroc"])
    for m in METHODS_H: ani_results[m].append(np.mean(trial[m]))
    print(f"  ratio={ratio:3d}  " + "  ".join(f"{m[:5]}={np.mean(trial[m]):.3f}" for m in METHODS_H))

fig, ax = plt.subplots(figsize=(8, 5))
for m in METHODS_H:
    lw=2.8 if m=="Deep-MELU" else 1.5
    ls="-" if m=="Deep-MELU" else "--"
    ax.semilogx(ratios, ani_results[m], color=COLORS[m], lw=lw, ls=ls,
                marker="o", ms=5, label=m)
ax.set_xlabel("Anisotropy ratio σ_max/σ_min (log scale)")
ax.set_ylabel("AUROC"); ax.set_ylim(0.3, 1.05)
ax.set_title("AUROC vs anisotropy ratio
"
             "(outliers extreme in lowest-variance dimension)")
ax.legend(fontsize=10); ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.savefig("outputs/anisotropy_experiment.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/anisotropy_experiment.png")


---
## Section D — AUCPR Focus

AUCPR is more discriminative than AUROC at low contamination. A method can have AUROC=0.95 but AUCPR=0.30 — the latter reflects what matters in practice: precision at the top of the score list.

## Cell 9 — PR curves on dependency anomalies

In [ ]:
from sklearn.metrics import precision_recall_curve

# Dependency anomalies: this is where Mahalanobis most clearly wins
X_dep, y_dep, name_dep, desc_dep = make_dependency_anomalies(n=800, dim=10, cont=0.05)
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X_dep)
X_in   = X_sc[y_dep == 0]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f"Precision-Recall Curves — {desc_dep}", fontsize=12)

score_store = {}
for m_name in METHODS_H:
    try:
        if m_name=="Deep-MELU":
            sc=DeepMELU(X_dep.shape[1]).fit(X_in,n_epochs=60).score(X_sc)
        elif m_name=="IsoForest":
            sc=-IsolationForest(200,contamination="auto",random_state=42).fit(X_in).score_samples(X_sc)
        elif m_name=="LOF":
            sc=-LocalOutlierFactor(20,novelty=True,contamination="auto").fit(X_in).score_samples(X_sc)
        elif m_name=="OC-SVM":
            sc=-OneClassSVM(kernel="rbf",nu=0.1,gamma="scale").fit(X_in).score_samples(X_sc)
        score_store[m_name]=sc
    except: score_store[m_name]=np.zeros(len(X_dep))

# Panel 1: PR curves
ax=axes[0]
for m_name in METHODS_H:
    sc=score_store[m_name]
    if np.isnan(sc).any(): continue
    prec,rec,_=precision_recall_curve(y_dep,sc)
    aucpr=average_precision_score(y_dep,sc)
    lw=2.8 if m_name=="Deep-MELU" else 1.5
    ax.plot(rec,prec,color=COLORS[m_name],lw=lw,
            label=f"{m_name} (AUCPR={aucpr:.3f})")
ax.axhline(y_dep.mean(),color="gray",lw=1,ls="--",
           label=f"Random ({y_dep.mean():.3f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall curve"); ax.legend(fontsize=9)
ax.grid(alpha=0.3); ax.set_xlim(0,1); ax.set_ylim(0,1.05)

# Panel 2: AUCPR bar with confidence
ax=axes[1]
aucpr_vals=[average_precision_score(y_dep,score_store[m]) for m in METHODS_H]
bars=ax.bar(METHODS_H,aucpr_vals,
            color=[COLORS[m] for m in METHODS_H],alpha=0.85)
ax.axhline(y_dep.mean(),color="gray",lw=1,ls="--",
           label=f"Random baseline ({y_dep.mean():.3f})")
for b,v in zip(bars,aucpr_vals):
    ax.text(b.get_x()+b.get_width()/2,v+0.01,f"{v:.3f}",
            ha="center",va="bottom",fontsize=10,fontweight="bold")
ax.set_ylabel("AUCPR (Average Precision)")
ax.set_title("AUCPR comparison"); ax.legend(fontsize=9)
ax.set_ylim(0,1.1); ax.grid(axis="y",alpha=0.3)
ax.set_xticklabels(METHODS_H,rotation=10)

plt.tight_layout()
plt.savefig("outputs/pr_curves_dependency.png",dpi=150,bbox_inches="tight")
plt.show()

print("Key insight for paper:")
dm_ap=average_precision_score(y_dep,score_store["Deep-MELU"])
best_baseline=max(average_precision_score(y_dep,score_store[m])
                  for m in ["IsoForest","LOF","OC-SVM"])
print(f"  Deep-MELU AUCPR: {dm_ap:.3f}")
print(f"  Best baseline:   {best_baseline:.3f}")
print(f"  Advantage:       {dm_ap-best_baseline:+.3f}")
print()
print("On dependency anomalies (broken correlation structure),")
print("Mahalanobis-based detection captures what Euclidean methods miss.")


## Cell 10 — Final summary figure for paper

In [ ]:
fig = plt.figure(figsize=(18, 11))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)
fig.suptitle("Deep-MELU Performance Profile — Hard Benchmarks for Pattern Recognition",
             fontsize=13)

# Subplot 1: hard datasets AUROC
ax1 = fig.add_subplot(gs[0, 0])
hard_aurocs = {m: [hard_results[n][m]["auroc"] for n in HARD_NAMES]
               for m in METHODS_H}
x = np.arange(len(HARD_NAMES)); w = 0.2
offsets = np.linspace(-1.5,1.5,len(METHODS_H))
for i,m in enumerate(METHODS_H):
    ax1.bar(x+offsets[i]*w, hard_aurocs[m], width=w,
            color=COLORS[m], alpha=0.87, label=m)
ax1.set_xticks(x)
ax1.set_xticklabels([n.replace("_","
")[:12] for n in HARD_NAMES],
                    fontsize=7)
ax1.set_ylabel("AUROC"); ax1.set_ylim(0.3,1.15)
ax1.set_title("Hard dataset AUROC"); ax1.legend(fontsize=7)
ax1.axhline(0.5,color="gray",lw=0.5,ls="--",alpha=0.4)

# Subplot 2: contamination robustness
ax2 = fig.add_subplot(gs[0, 1])
for m in METHODS_H:
    lw=2.5 if m=="Deep-MELU" else 1.3
    ls="-" if m=="Deep-MELU" else "--"
    ax2.plot(x_cont, sweep_results[m]["auroc"],
             color=COLORS[m],lw=lw,ls=ls,marker="o",ms=4,label=m)
ax2.axvline(25,color="gray",lw=1,ls=":",alpha=0.6)
ax2.fill_betweenx([0.3,1.05],0,25,alpha=0.05,color="#1D9E75")
ax2.set_xlabel("Contamination %"); ax2.set_ylabel("AUROC")
ax2.set_title("Contamination robustness (MCD safe zone shaded)")
ax2.legend(fontsize=8); ax2.set_ylim(0.3,1.05); ax2.grid(alpha=0.25)

# Subplot 3: correlation experiment
ax3 = fig.add_subplot(gs[0, 2])
for m in METHODS_H:
    lw=2.5 if m=="Deep-MELU" else 1.3
    ls="-" if m=="Deep-MELU" else "--"
    ax3.plot(rho_values, rho_results[m],
             color=COLORS[m],lw=lw,ls=ls,marker="o",ms=4,label=m)
ax3.set_xlabel("Correlation ρ"); ax3.set_ylabel("AUROC")
ax3.set_title("Correlation strength experiment")
ax3.legend(fontsize=8); ax3.set_ylim(0.3,1.05); ax3.grid(alpha=0.25)

# Subplot 4: anisotropy experiment
ax4 = fig.add_subplot(gs[1, 0])
for m in METHODS_H:
    lw=2.5 if m=="Deep-MELU" else 1.3
    ls="-" if m=="Deep-MELU" else "--"
    ax4.semilogx(ratios, ani_results[m],
                 color=COLORS[m],lw=lw,ls=ls,marker="o",ms=4,label=m)
ax4.set_xlabel("Anisotropy ratio"); ax4.set_ylabel("AUROC")
ax4.set_title("Anisotropy experiment (log scale)")
ax4.legend(fontsize=8); ax4.set_ylim(0.3,1.05); ax4.grid(alpha=0.25,which="both")

# Subplot 5: Hard dataset AUCPR
ax5 = fig.add_subplot(gs[1, 1])
hard_aucprs = {m: [hard_results[n][m]["aucpr"] for n in HARD_NAMES]
               for m in METHODS_H}
for i,m in enumerate(METHODS_H):
    ax5.bar(x+offsets[i]*w, hard_aucprs[m], width=w,
            color=COLORS[m], alpha=0.87, label=m)
ax5.set_xticks(x)
ax5.set_xticklabels([n.replace("_","
")[:12] for n in HARD_NAMES], fontsize=7)
ax5.set_ylabel("AUCPR"); ax5.set_ylim(0,1.15)
ax5.set_title("Hard dataset AUCPR"); ax5.legend(fontsize=7)

# Subplot 6: Summary text
ax6 = fig.add_subplot(gs[1, 2]); ax6.axis("off")
summary_lines = []
for name in HARD_NAMES:
    dm_a = hard_results[name]["Deep-MELU"]["auroc"]
    best_bl = max(hard_results[name][m]["auroc"] for m in ["IsoForest","LOF","OC-SVM"])
    delta = dm_a - best_bl
    icon = "✓" if delta >= 0 else "✗"
    summary_lines.append(f"{icon}  {name[:22]:<22}  Δ={delta:+.3f}")

ax6.text(0.05, 0.95, "Deep-MELU vs best baseline (AUROC Δ)",
         transform=ax6.transAxes, fontsize=10, fontweight="bold", va="top")
for i, line in enumerate(summary_lines):
    color="#085041" if "✓" in line else "#993C1D"
    ax6.text(0.05, 0.82-i*0.13, line, transform=ax6.transAxes,
             fontsize=9, va="top", color=color, fontfamily="monospace")

plt.savefig("outputs/paper_figure_hard_benchmarks.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Paper figure saved → outputs/paper_figure_hard_benchmarks.png")

# Save all results
rows=[]
for name in HARD_NAMES:
    for m in METHODS_H:
        rows.append({"Dataset":name,"Method":m,
                     "AUROC":round(hard_results[name][m]["auroc"],4),
                     "AUCPR":round(hard_results[name][m]["aucpr"],4)})
pd.DataFrame(rows).to_csv("outputs/hard_benchmark_results.csv",index=False)
print("CSV saved → outputs/hard_benchmark_results.csv")
